<a href="https://colab.research.google.com/github/huimouqianxiao123/Peft-Qwen2.5-/blob/main/test_UnQLora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U transformers accelerate bitsandbytes huggingface_hub peft scikit-learn tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 153.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 156.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 9.6 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.18.0
    Uninstalling hugging

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
model_path = "/content/drive/MyDrive/models/qwen2.5-7b-instruct"
adapter_path = ""  # 如果要测试 LoRA/PEFT adapter，填入 adapter 目录；不需要则保持空字符串
file_path = "/content/drive/MyDrive/data/intent_train_multiturn_qwen.jsonl"

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

if adapter_path:
    from peft import PeftModel

    model = PeftModel.from_pretrained(model, adapter_path)

model.eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [5]:
import json
import random

all_data = []
skipped_rows = 0

print(f"正在读取数据集: {file_path}")
with open(file_path, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():
            continue

        item = json.loads(line)
        messages = item.get("messages", [])
        if len(messages) < 2 or messages[-1].get("role") != "assistant" or "content" not in messages[-1]:
            skipped_rows += 1
            continue

        all_data.append(item)

if not all_data:
    raise ValueError("没有读到可评估样本，请检查 file_path 和 JSONL 数据格式。")

sample_size = min(500, len(all_data))
random.seed(42)
sampled_dataset = random.sample(all_data, sample_size)

print(f"可评估样本数: {len(all_data)}")
print(f"跳过异常样本数: {skipped_rows}")
print(f"成功随机抽取 {sample_size} 条测试样本。")

正在读取数据集: /content/drive/MyDrive/data/intent_train_multiturn_qwen.jsonl
可评估样本数: 5000
跳过异常样本数: 0
成功随机抽取 500 条测试样本。


In [6]:
from tqdm.auto import tqdm

intents_list = [
    "查询物流", "催促配送", "修改订单", "取消订单", "申请退款", "退换货",
    "发票咨询", "支付咨询", "价格价保", "优惠活动", "商品咨询", "催促处理",
    "投诉反馈", "转人工", "问候确认", "业务咨询", "查询订单", "其他",
]


def normalize_label(response_text):
    text = response_text.strip()
    for intent in intents_list:
        if text == intent or intent in text:
            return intent
    return "其他"


y_true = []
y_pred = []
raw_outputs = []

print("开始执行 500 条随机样本的意图识别准确率测试...")
for item in tqdm(sampled_dataset):
    messages = item.get("messages", [])
    prompt_messages = messages[:-1]
    true_label = messages[-1]["content"].strip()

    if true_label not in intents_list:
        continue

    text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    output_ids = generated_ids[0][model_inputs.input_ids.shape[-1]:]
    response = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
    pred_label = normalize_label(response)

    y_true.append(true_label)
    y_pred.append(pred_label)
    raw_outputs.append({
        "true_label": true_label,
        "pred_label": pred_label,
        "raw_response": response,
        "current_user_utterance": item.get("metadata", {}).get("current_user_utterance", ""),
    })

开始执行 500 条随机样本的意图识别准确率测试...


  0%|          | 0/500 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [7]:
from sklearn.metrics import accuracy_score, classification_report

if not y_true:
    raise ValueError("没有有效评估结果，请检查标签集合或数据格式。")

accuracy = accuracy_score(y_true, y_pred)

print("\n================== 测试评估报告 ==================")
print(f"有效评估样本数: {len(y_true)}")
print(f"总体准确率 (Overall Accuracy): {accuracy:.2%}\n")
print("各意图类别的详细指标 (Precision / Recall / F1-Score):")
print(classification_report(y_true, y_pred, labels=intents_list, zero_division=0))
print("==================================================")


================== 测试评估报告 ==================
有效评估样本数: 500
总体准确率 (Overall Accuracy): 19.80%

各意图类别的详细指标 (Precision / Recall / F1-Score):
              precision    recall  f1-score   support

        查询物流       0.37      0.28      0.32        54
        催促配送       0.00      0.00      0.00         0
        修改订单       0.18      0.60      0.27         5
        取消订单       0.08      0.67      0.14         3
        申请退款       0.36      0.22      0.28        18
         退换货       0.45      0.48      0.47        31
        发票咨询       0.25      1.00      0.40         5
        支付咨询       0.20      0.20      0.20         5
        价格价保       0.55      0.75      0.63         8
        优惠活动       0.50      0.75      0.60         4
        商品咨询       0.05      0.50      0.08         6
        催促处理       0.03      0.67      0.06         3
        投诉反馈       0.00      0.00      0.00         0
         转人工       0.00      0.00      0.00         0
        问候确认       0.64      0.21      0.32       10